# Aula 07 — LLMOps: a jornada de um agente em produção (Databricks)

Notebook interativo da Aula 07. Construímos um **agente de suporte** sobre os primitivos nativos do
**Databricks Agent Framework** e percorremos o ciclo de LLMOps:

`SETUP → DESENVOLVER → VERSIONAR → FEEDBACK → GATE → SERVIR`

Cobrimos: **tools como UC Functions**, **RAG por Vector Search**, **observabilidade com autolog**,
**feedback humano**, **prompt registry**, **gate de aprovação (testes)** e **deploy em Model Serving**.

> Roda **dentro de um workspace Databricks** (precisa da sessão `spark` no setup e de credenciais).
> Os **guardrails** e os **painéis de monitoramento** ficam na UI do workspace.

## Ambiente
Instala as bibliotecas atuais e reinicia o Python (no Databricks).

In [ ]:
%pip install -U mlflow databricks-langchain databricks-vectorsearch unitycatalog-ai databricks-agents pytest

In [ ]:
try:
    dbutils.library.restartPython()  # noqa: F821
except NameError:
    pass

In [ ]:
import os, sys
try:
    _dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _dir = os.getcwd()
while _dir != os.path.dirname(_dir):
    if os.path.isdir(os.path.join(_dir, 'agente')):
        BASE = _dir; sys.path.insert(0, _dir); break
    _dir = os.path.dirname(_dir)

from agente.config import CONFIG_PADRAO, PROMPT_UC, MODELO_UC, INDICE_POLITICAS, LOJA
from agente.mlflow_databricks import credenciais_ok, configurar_mlflow

assert credenciais_ok(), 'Configure DATABRICKS_HOST/TOKEN no aula-07/.env (veja o .env.example).'
configurar_mlflow()
print('endpoint:', CONFIG_PADRAO.endpoint, '| índice:', INDICE_POLITICAS)

## Passo 0 — Provisionar os recursos no Unity Catalog

Idempotente: registra as **UC Functions** (tools) e cria a tabela de políticas + o **índice de
Vector Search** (RAG). A criação do índice leva alguns minutos na primeira vez.

In [ ]:
from agente.config import CATALOGO, SCHEMA
from agente.ferramentas import registrar_uc_functions
from agente.base_vetorial import criar_tabela_politicas, criar_indice, POLITICAS

# Tudo mora em aula_07.default — criamos o catálogo/schema (idempotente) antes de qualquer coisa.
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOGO}')          # noqa: F821 (spark é global)
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}')  # noqa: F821
print('catálogo/schema:', f'{CATALOGO}.{SCHEMA}')

print('UC Functions:', registrar_uc_functions())
tabela = criar_tabela_politicas(spark)  # noqa: F821 (spark é global no Databricks)
print('tabela de políticas:', tabela, f'({len(POLITICAS)} chunks)')
print('índice Vector Search:', criar_indice())

## Passo 1 — Desenvolver o agente (tools + RAG + observabilidade)

O `responder()` roda o agente (ChatDatabricks + UC Functions + retriever de Vector Search) com
`mlflow.langchain.autolog()` ligado: cada requisição vira uma **trace**.

In [ ]:
from agente.agente_suporte import responder, TELEMETRIA

roteiro = [
    'Qual o status do pedido 1002?',                 # UC Function
    'Quanto fica o frete para o CEP 01000-000, 2 kg?',  # UC Function
    'Posso trocar um produto comprado há 20 dias?',  # RAG (Vector Search)
    'Qual a garantia dos eletrônicos?',              # RAG (Vector Search)
]
for p in roteiro:
    r = responder(p)
    print(f'\nQ: {p}\nA: {r.texto}\n   ferramentas: {r.ferramentas_usadas} | {r.latencia_ms:.0f} ms | trace {r.trace_id}')

### Observabilidade
Cada requisição acima virou uma **trace** (aba *Traces* no MLflow). Abaixo, um resumo da sessão.

In [ ]:
tools = [f for t in TELEMETRIA for f in t.ferramentas_usadas]
print('requisições :', len(TELEMETRIA))
print('chamadas tool:', len(tools), sorted(set(tools)))
print('>> abra a aba Traces no MLflow para inspecionar os spans.')

## Passo 2 — Versionar o prompt (MLflow Prompt Registry)

O prompt é o artefato que mais muda. Registramos versões **imutáveis** com `{{variáveis}}` e
movemos aliases (`champion`/`challenger`).

In [ ]:
import mlflow

def reg(tpl, autor):
    return mlflow.genai.register_prompt(name=PROMPT_UC, template=tpl,
        commit_message=f'versão por {autor}', tags={'autor': autor})

v1 = reg('Você é o atendente da loja {{loja}}. Responda em 1 frase.', 'luiz')
v2 = reg(CONFIG_PADRAO.system_prompt.replace(LOJA, '{{loja}}'), 'ana')
mlflow.genai.set_prompt_alias(PROMPT_UC, 'champion', v1.version)
mlflow.genai.set_prompt_alias(PROMPT_UC, 'challenger', v2.version)
champ = mlflow.genai.load_prompt(f'prompts:/{PROMPT_UC}@champion')
print('champion -> v', champ.version, '|', champ.format(loja=LOJA)[:80], '...')
# challenger venceu -> promove
mlflow.genai.set_prompt_alias(PROMPT_UC, 'champion', v2.version)
print('após promover, champion -> v', mlflow.genai.load_prompt(f'prompts:/{PROMPT_UC}@champion').version)

## Passo 3 — Feedback humano

O 👍/👎 do usuário é gravado com `mlflow.log_feedback` como *assessment* HUMAN ligada à trace.

In [ ]:
from agente.agente_suporte import registrar_feedback

r = responder('Qual a garantia dos eletrônicos?')
registrar_feedback(r, util=True, comentario='respondeu certo: 12 meses', usuario='cliente@acme.com')
print('👍 feedback gravado na trace', r.trace_id)

## Passo 4 — Gate de aprovação (testes unitários)

O gate é um `pytest` sobre situações conhecidas + recusa de jailbreak. *Exit code* != 0 bloqueia o
merge no CI. Rodamos o mesmo arquivo do CI.

In [ ]:
import pytest
codigo = pytest.main([os.path.join(BASE, 'src', '04_gate_aprovacao.py'), '-v', '-ra'])
print('\n>> gate', 'VERDE ✅' if codigo == 0 else f'VERMELHO ❌ (exit {int(codigo)})')

## Passo 5 — Servir o agente (deploy real)

`log_model` (com `resources`) → registro no Unity Catalog → `agents.deploy`. Leva alguns minutos.
Descomente para provisionar o endpoint.

In [ ]:
# from mlflow.models.resources import (DatabricksServingEndpoint,
#     DatabricksVectorSearchIndex, DatabricksFunction)
# from agente.config import (ENDPOINT_LLM, ENDPOINT_EMBEDDINGS, CATALOGO, SCHEMA,
#     FUNC_STATUS_PEDIDO, FUNC_CALCULAR_FRETE)
# input_example = {'input': [{'role': 'user', 'content': 'Qual a garantia dos eletrônicos?'}]}
# with mlflow.start_run():
#     info = mlflow.pyfunc.log_model(
#         name='agente_suporte', python_model=os.path.join(BASE, 'agente', 'serving_model.py'),
#         code_paths=[os.path.join(BASE, 'agente')], input_example=input_example,
#         resources=[DatabricksServingEndpoint(endpoint_name=ENDPOINT_LLM),
#                    DatabricksServingEndpoint(endpoint_name=ENDPOINT_EMBEDDINGS),
#                    DatabricksVectorSearchIndex(index_name=INDICE_POLITICAS),
#                    DatabricksFunction(function_name=FUNC_STATUS_PEDIDO),
#                    DatabricksFunction(function_name=FUNC_CALCULAR_FRETE)],
#         registered_model_name=MODELO_UC,
#         pip_requirements=['mlflow','databricks-langchain','databricks-vectorsearch','unitycatalog-ai'])
# from databricks import agents
# # environment_vars leva os endpoints/catálogo p/ o container (lá não há .env; senão, 404 no default)
# dep = agents.deploy(MODELO_UC, info.registered_model_version, scale_to_zero=True,
#                     environment_vars={'ENDPOINT_LLM_AGENTE': ENDPOINT_LLM,
#                                       'ENDPOINT_EMBEDDINGS': ENDPOINT_EMBEDDINGS,
#                                       'UC_CATALOG': CATALOGO, 'UC_SCHEMA': SCHEMA})
# print('endpoint:', dep.endpoint_name)
print('Passo 5: veja aula-07/src/05_serving_deploy.py para o deploy completo.')

## Conclusão

Percorremos o ciclo de LLMOps com governança: recursos no **Unity Catalog** (UC Functions + Vector
Search), **observabilidade** por autolog, **prompt registry**, **feedback humano**, **gate de
aprovação** por testes e **deploy** em Model Serving — a mesma esteira que o **GitHub Actions**
(`ci/ci-cd-agente.yml`) automatiza.